# Prompt-Level Compressibility

This notebook asks what can be predicted before the reasoning trace is observed.

Unit of analysis: one full trace.

Target: `trace_high_token_compression`, created in `01_data_loading_engineering.ipynb` from the training-split trace compression threshold.

Predictors: problem-derived features and metadata only.

Models: dummy baseline, random forest, and gradient boosting. Hyperparameters are selected on a grouped validation split taken only from the training split, then the selected model is refit on the full training split and evaluated once on the test split.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

sys.path.append(str(Path("..").resolve()))

from src.reasoning_compression.features import (
    count_share_table,
    format_metrics_percent,
    latest_feature_build_dir,
    normalize_difficulty,
)

from src.reasoning_compression.modeling import (
    GRADIENT_BOOSTING_PARAM_GRID,
    RANDOM_FOREST_PARAM_GRID,
    classification_report_frame,
    evaluate_classifier,
    fit_selected_model,
    make_preprocessor,
    random_forest_feature_importance,
    select_model_params,
)

In [2]:
FULL_BUILD_DIR = latest_feature_build_dir(Path("../data/full_feature_builds"))
TRACE_TARGET = "trace_high_token_compression"
SPLIT_COL = "model_split"
GROUP_COL = "trace_id"

df_traces = pd.read_parquet(
    FULL_BUILD_DIR / "traces_features_full_labeled.parquet"
)
df_traces["difficulty"] = df_traces["difficulty"].map(normalize_difficulty)

required_columns = {TRACE_TARGET, SPLIT_COL, GROUP_COL}
missing_columns = required_columns.difference(df_traces.columns)
if missing_columns:
    raise ValueError(
        "Trace table is missing required columns: "
        f"{sorted(missing_columns)}. Re-run 01_data_loading_engineering.ipynb "
        "with RUN_FULL_BUILD = False."
    )

df_traces.shape

(228557, 27)

In [3]:
split_target_distribution = (
    df_traces
    .groupby(SPLIT_COL)[TRACE_TARGET]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .assign(share_pct=lambda df: (df["share"] * 100).round(2))
    .drop(columns="share")
)

split_target_distribution

,model_split,trace_high_token_compression,share_pct
0,test,0,75.12
1,test,1,24.88
2,train,0,75.00
3,train,1,25.00


**Prompt split check interpretation**

The target distribution is behaving as expected. The training split is exactly anchored at the 25% high-compression threshold, and the held-out test split is very close to the same class balance. This supports using balanced accuracy and ROC AUC as the main evaluation metrics rather than raw accuracy alone.

In [4]:
count_share_table(df_traces, "domain")

,n_rows,share_pct
domain,,
math,123333,53.96
science,61485,26.90
code,43739,19.14


## Feature Set

Prompt-level models use only information available before the model begins producing the reasoning trace: problem features and metadata.

In [5]:
numeric_features = [
    "problem_chars",
    "problem_tokens",
    "problem_math_symbol_share",
    "problem_question_mark_count",
]

binary_features = [
    "problem_has_multiple_choice",
    "problem_has_code_fence",
]

categorical_features = [
    "domain",
    "source",
    "difficulty",
]

feature_columns = numeric_features + binary_features + categorical_features

In [6]:
df_model = df_traces[
    feature_columns + [TRACE_TARGET, GROUP_COL, SPLIT_COL]
].dropna().copy()

X = df_model[feature_columns]
y = df_model[TRACE_TARGET]
groups = df_model[GROUP_COL]
model_split = df_model[SPLIT_COL]

train_mask = model_split == "train"
test_mask = model_split == "test"

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]
groups_train = groups.loc[train_mask]
groups_test = groups.loc[test_mask]

X_train.shape, X_test.shape

((182845, 9), (45712, 9))

In [7]:
len(set(groups_train).intersection(set(groups_test)))

0

## Modeling Utilities

Shared evaluation, grouped validation tuning, and feature-importance helpers are imported from `src.reasoning_compression.modeling`.


In [ ]:
rf_param_grid = RANDOM_FOREST_PARAM_GRID
gb_param_grid = GRADIENT_BOOSTING_PARAM_GRID

In [10]:
preprocessor = make_preprocessor(
    numeric_columns=numeric_features,
    binary_columns=binary_features,
    categorical_columns=categorical_features,
)

## Baseline

In [11]:
prompt_dummy_model = DummyClassifier(strategy="most_frequent")
prompt_dummy_model.fit(X_train, y_train)

prompt_dummy_metrics = evaluate_classifier(prompt_dummy_model, X_test, y_test)
format_metrics_percent(prompt_dummy_metrics)

,metric,value
0,accuracy,75.12%
1,balanced_accuracy,50.00%
2,roc_auc,50.00%


## Random Forest

In [12]:
prompt_rf_selection_results = select_model_params(
    model_name="Random forest",
    model_class=RandomForestClassifier,
    param_grid=rf_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

prompt_rf_selection_results

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",30000,24000,6000,0.617167,0.574917,0.619909
1,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",30000,24000,6000,0.583833,0.577186,0.619185
2,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",30000,24000,6000,0.618500,0.577481,0.618691
3,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",30000,24000,6000,0.583167,0.572646,0.617521


In [13]:
prompt_rf_params = prompt_rf_selection_results.loc[0, "params"]
prompt_rf_model = fit_selected_model(
    model_class=RandomForestClassifier,
    selected_params=prompt_rf_params,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
)

prompt_rf_pred = prompt_rf_model.predict(X_test)
prompt_rf_metrics = evaluate_classifier(prompt_rf_model, X_test, y_test)
format_metrics_percent(prompt_rf_metrics)

,metric,value
0,accuracy,62.94%
1,balanced_accuracy,62.47%
2,roc_auc,68.07%


In [14]:
random_forest_feature_importance(prompt_rf_model).head(20).round(4)

,feature,importance
0,num__problem_chars,0.3263
1,num__problem_math_symbol_share,0.2982
2,num__problem_tokens,0.2486
3,num__problem_question_mark_count,0.0312
4,cat__source_ai2-adapt-dev/openmath-2-math,0.0198
5,cat__domain_science,0.0197
6,cat__domain_math,0.0196
7,cat__source_stackexchange-physics,0.0118
8,cat__source_organic-chemistry-questions,0.0057
9,bin__problem_has_multiple_choice,0.0041


**Prompt feature importance interpretation**

The random forest relies mostly on prompt length and prompt composition: character count, token count, and math-symbol share dominate. Metadata and difficulty contribute much less. This is consistent with the prompt-level task: before any reasoning block is observed, the available signal is mostly about the type and surface structure of the problem.

## Gradient Boosting

In [15]:
prompt_gb_selection_results = select_model_params(
    model_name="Gradient boosting",
    model_class=HistGradientBoostingClassifier,
    param_grid=gb_param_grid,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
)

prompt_gb_selection_results

,model,params,n_tuning_rows,n_fit_rows,n_validation_rows,accuracy,balanced_accuracy,roc_auc
0,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.527000,0.586002,0.614361
1,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",30000,24000,6000,0.761833,0.500396,0.611506
2,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.530667,0.585511,0.611009
3,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",30000,24000,6000,0.762333,0.500483,0.610828
4,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.503667,0.581557,0.610309
5,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.504333,0.578375,0.609343
6,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.510333,0.581586,0.609253
7,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.515500,0.582079,0.609224
8,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",30000,24000,6000,0.511833,0.579433,0.609210
9,Gradient boosting,"{'class_weight': None, 'l2_regularization': 0....",30000,24000,6000,0.762333,0.500000,0.609068


In [16]:
prompt_gb_params = prompt_gb_selection_results.loc[0, "params"]
prompt_gb_model = fit_selected_model(
    model_class=HistGradientBoostingClassifier,
    selected_params=prompt_gb_params,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
)

prompt_gb_pred = prompt_gb_model.predict(X_test)
prompt_gb_metrics = evaluate_classifier(prompt_gb_model, X_test, y_test)
format_metrics_percent(prompt_gb_metrics)

,metric,value
0,accuracy,53.12%
1,balanced_accuracy,60.69%
2,roc_auc,64.44%


## Model Comparison

In [17]:
prompt_model_results = pd.DataFrame([
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Dummy",
        "selected_params": None,
        **prompt_dummy_metrics,
    },
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Random forest",
        "selected_params": prompt_rf_params,
        **prompt_rf_metrics,
    },
    {
        "task": "Prompt-level compressibility",
        "feature_set": "Problem + metadata only",
        "model": "Gradient boosting",
        "selected_params": prompt_gb_params,
        **prompt_gb_metrics,
    },
])

prompt_model_results_display = prompt_model_results.copy()
for metric in ["accuracy", "balanced_accuracy", "roc_auc"]:
    prompt_model_results_display[metric] = (
        prompt_model_results_display[metric] * 100
    ).round(2)

prompt_model_results_display

,task,feature_set,model,selected_params,accuracy,balanced_accuracy,roc_auc
0,Prompt-level compressibility,Problem + metadata only,Dummy,None,75.12,50.00,50.00
1,Prompt-level compressibility,Problem + metadata only,Random forest,"{'class_weight': 'balanced', 'min_samples_leaf...",62.94,62.47,68.07
2,Prompt-level compressibility,Problem + metadata only,Gradient boosting,"{'class_weight': 'balanced', 'l2_regularizatio...",53.12,60.69,64.44


**Prompt model comparison interpretation**

The prompt-only models improve clearly over the dummy baseline in balanced accuracy and ROC AUC, but the effect is modest. Random forest is the strongest model in this run, with ROC AUC around 68%, while gradient boosting is weaker but has higher recall for the high-compression class. This is the expected pattern for the prompt-level question: the problem statement contains some predictive signal, but not enough to strongly determine compressibility before the reasoning trace is generated.

In [18]:
prompt_reports = pd.concat(
    [
        classification_report_frame("Random forest", y_test, prompt_rf_pred),
        classification_report_frame("Gradient boosting", y_test, prompt_gb_pred),
    ],
    ignore_index=True,
)

prompt_reports.round(4)

,model,label,precision,recall,f1-score,support
0,Random forest,0,0.8327,0.6340,0.7199,34338.0
1,Random forest,1,0.3577,0.6154,0.4524,11374.0
2,Random forest,accuracy,NaN,NaN,0.6294,45712.0
3,Random forest,macro avg,0.5952,0.6247,0.5861,45712.0
4,Random forest,weighted avg,0.7145,0.6294,0.6533,45712.0
5,Gradient boosting,0,0.8503,0.4563,0.5939,34338.0
6,Gradient boosting,1,0.3158,0.7575,0.4457,11374.0
7,Gradient boosting,accuracy,NaN,NaN,0.5312,45712.0
8,Gradient boosting,macro avg,0.5830,0.6069,0.5198,45712.0
9,Gradient boosting,weighted avg,0.7173,0.5312,0.5570,45712.0


**Prompt classification report interpretation**

The classification reports show the practical tradeoff behind the aggregate metrics. Random forest is more balanced overall, while gradient boosting recovers more high-compression traces but at the cost of many false positives and lower raw accuracy. For the prompt-only task, these predictions should be read as weak early risk scores rather than reliable classifications.